# Notebook 1 — BioGPT Attention Graph from MedMCQA

This clean notebook:

1. Loads one MedMCQA validation question from its standard Parquet file.
2. Builds a clinical multiple-choice prompt without revealing the correct answer.
3. Extracts BioGPT self-attention.
4. Aggregates layers, heads, and BioGPT subwords.
5. Constructs a sparse directed weighted attention graph using the strongest outgoing edges.
6. Exports the graph, matrices, tables, metadata, and figures required by Notebook 2.

**Runtime:** Google Colab with a T4 GPU.

## 1. Install only the required packages

In [ ]:
# Keep Colab's existing NumPy, pandas, and PyTorch installations.
# This avoids package conflicts that can damage the runtime.

!pip -q install "transformers>=4.57,<5" "datasets>=4.0,<5" accelerate sentencepiece sacremoses networkx openpyxl

## 2. Imports and environment verification

In [ ]:
import gc
import json
import pickle
import platform
import random
import shutil
import string
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not detected. In Colab select Runtime → Change runtime type → T4 GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))

## 3. Reproducibility and experiment settings

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda")

MODEL_NAME = "microsoft/biogpt"
DATASET_NAME = "openlifescienceai/medmcqa"
DATASET_SPLIT = "validation"
SAMPLE_INDEX = 0

MAX_LENGTH = 256
TOP_K_EDGES_PER_NODE = 5
REMOVE_SELF_LOOPS = True
REMOVE_SPECIAL_TOKENS = True
REMOVE_PUNCTUATION_ONLY_TOKENS = True

OUTPUT_DIR = Path("/content/notebook1_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Model:", MODEL_NAME)
print("Dataset:", DATASET_NAME)
print("Split:", DATASET_SPLIT)
print("Sample index:", SAMPLE_INDEX)
print("Output directory:", OUTPUT_DIR)

## 4. Load MedMCQA from a standard Parquet file

In [ ]:
# The explicit Parquet URI avoids all legacy dataset-script and trust_remote_code issues.
MEDMCQA_VALIDATION_PARQUET = (
    "hf://datasets/openlifescienceai/medmcqa/"
    "data/validation-00000-of-00001.parquet"
)

print("Loading:", MEDMCQA_VALIDATION_PARQUET)

dataset = load_dataset(
    "parquet",
    data_files={"validation": MEDMCQA_VALIDATION_PARQUET},
    split="validation"
)

print("Dataset loaded successfully.")
print("Number of validation samples:", len(dataset))
print("Columns:", dataset.column_names)

## 5. Inspect the selected clinical question

In [ ]:
sample = dataset[SAMPLE_INDEX]

required_fields = [
    "id", "question", "opa", "opb", "opc", "opd", "cop",
    "choice_type", "exp", "subject_name", "topic_name"
]

missing_fields = [field for field in required_fields if field not in sample]
if missing_fields:
    raise KeyError(f"Missing MedMCQA fields: {missing_fields}")

options = {
    "A": str(sample["opa"]),
    "B": str(sample["opb"]),
    "C": str(sample["opc"]),
    "D": str(sample["opd"])
}

correct_index = int(sample["cop"])
if correct_index not in range(4):
    raise ValueError(f"Unexpected correct-option index: {correct_index}")

correct_letter = ["A", "B", "C", "D"][correct_index]
correct_answer = options[correct_letter]

print("Question:", sample["question"])
print()
for letter, option in options.items():
    print(f"{letter}. {option}")

print("\nGround truth:", correct_letter, "-", correct_answer)
print("Subject:", sample["subject_name"])
print("Topic:", sample["topic_name"])

## 6. Build the BioGPT prompt

In [ ]:
def build_prompt(question, answer_options):
    lines = [
        "Clinical question:",
        str(question).strip(),
        "",
        "Possible answers:"
    ]

    for letter in ["A", "B", "C", "D"]:
        lines.append(f"{letter}. {answer_options[letter].strip()}")

    lines.extend([
        "",
        "Select the most appropriate answer and explain the clinical reasoning."
    ])

    return "\n".join(lines)


prompt = build_prompt(sample["question"], options)
print(prompt)

## 7. Save case metadata

In [ ]:
metadata = {
    "model_name": MODEL_NAME,
    "dataset_name": DATASET_NAME,
    "dataset_split": DATASET_SPLIT,
    "sample_index": SAMPLE_INDEX,
    "question_id": str(sample["id"]),
    "question": str(sample["question"]),
    "options": options,
    "correct_option_index_zero_based": correct_index,
    "correct_option_letter": correct_letter,
    "correct_answer": correct_answer,
    "choice_type": str(sample["choice_type"]),
    "expert_explanation": str(sample["exp"]),
    "subject_name": str(sample["subject_name"]),
    "topic_name": str(sample["topic_name"]),
    "max_length": MAX_LENGTH,
    "top_k_edges_per_node": TOP_K_EDGES_PER_NODE,
    "prompt": prompt
}

metadata_path = OUTPUT_DIR / "metadata.json"
with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2, ensure_ascii=False)

print("Saved:", metadata_path)

## 8. Load BioGPT with eager attention

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    attn_implementation="eager",
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True
)

model.config.output_attentions = True
model.config.use_cache = False
model.config.return_dict = True

model.to(DEVICE)
model.eval()

print("Model class:", model.__class__.__name__)
print("Layers:", model.config.num_hidden_layers)
print("Attention heads:", model.config.num_attention_heads)
print("Hidden size:", model.config.hidden_size)
print("Device:", next(model.parameters()).device)

## 9. Tokenize the prompt

In [ ]:
encoded = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_LENGTH,
    add_special_tokens=True
)

input_ids_cpu = encoded["input_ids"][0].clone()
subword_tokens = tokenizer.convert_ids_to_tokens(input_ids_cpu.tolist())
model_inputs = {key: value.to(DEVICE) for key, value in encoded.items()}

subword_table = pd.DataFrame({
    "subword_index": np.arange(len(subword_tokens)),
    "subword_token": subword_tokens,
    "token_id": input_ids_cpu.tolist()
})

print("Sequence length:", len(subword_tokens))
print("Reached maximum length:", len(subword_tokens) == MAX_LENGTH)
display(subword_table.head(40))

## 10. Extract and average BioGPT attention

In [ ]:
start_time = time.time()

with torch.inference_mode():
    outputs = model(
        **model_inputs,
        output_attentions=True,
        use_cache=False,
        return_dict=True
    )

elapsed_seconds = time.time() - start_time

if outputs.attentions is None:
    raise RuntimeError(
        "No attention tensors were returned. Confirm that the model was loaded "
        "with attn_implementation='eager'."
    )

print("Number of returned layers:", len(outputs.attentions))
print("One layer shape:", tuple(outputs.attentions[0].shape))
print("Extraction time:", round(elapsed_seconds, 3), "seconds")

In [ ]:
# Average heads inside each layer, then average all layers.
# Moving each layer mean to CPU prevents unnecessary GPU memory accumulation.

layer_means = []

for layer_attention in outputs.attentions:
    # Shape before averaging: batch × heads × query × key
    layer_mean = layer_attention[0].float().mean(dim=0).cpu()
    layer_means.append(layer_mean)

mean_subword_attention = (
    torch.stack(layer_means, dim=0)
    .mean(dim=0)
    .numpy()
    .astype(np.float64)
)

expected_shape = (len(subword_tokens), len(subword_tokens))
assert mean_subword_attention.shape == expected_shape
assert np.isfinite(mean_subword_attention).all()
assert (mean_subword_attention >= 0).all()

print("Mean subword-attention shape:", mean_subword_attention.shape)
print("Minimum weight:", mean_subword_attention.min())
print("Maximum weight:", mean_subword_attention.max())
print("Mean row sum:", mean_subword_attention.sum(axis=1).mean())

## 11. Check causal attention direction

In [ ]:
future_mass = np.triu(mean_subword_attention, k=1).sum()
total_mass = mean_subword_attention.sum()
future_mass_ratio = future_mass / total_mass if total_mass else np.nan

print("Future-attention mass ratio:", future_mass_ratio)

if future_mass_ratio > 1e-5:
    print("Warning: inspect the attention tensors before continuing.")
else:
    print("Causal-direction check passed.")

## 12. Merge BioGPT subwords into readable tokens

In [ ]:
SPECIAL_TOKENS = set(tokenizer.all_special_tokens)


def merge_biogpt_subwords(tokens):
    merged_tokens = []
    groups = []
    current_parts = []
    current_indices = []

    def close_current():
        nonlocal current_parts, current_indices
        if current_indices:
            text = "".join(current_parts).strip()
            if not text:
                text = "".join(str(tokens[index]) for index in current_indices)
            merged_tokens.append(text)
            groups.append(current_indices.copy())
        current_parts = []
        current_indices = []

    for index, raw_token in enumerate(tokens):
        token = str(raw_token)

        if token in SPECIAL_TOKENS:
            close_current()
            merged_tokens.append(token)
            groups.append([index])
            continue

        # BioGPT/Fairseq end-of-word marker.
        if token.endswith("</w>"):
            current_parts.append(token[:-4])
            current_indices.append(index)
            close_current()
            continue

        # Safeguards for tokenizers using alternative word-boundary markers.
        if token.startswith("Ġ") or token.startswith("▁"):
            close_current()
            current_parts = [token[1:]]
            current_indices = [index]
            continue

        if token.startswith("##"):
            current_parts.append(token[2:])
            current_indices.append(index)
            continue

        current_parts.append(token)
        current_indices.append(index)

    close_current()

    flattened = [index for group in groups for index in group]
    if sorted(flattened) != list(range(len(tokens))):
        raise RuntimeError("Token merging lost or duplicated subword indices.")

    return merged_tokens, groups


merged_tokens, subword_groups = merge_biogpt_subwords(subword_tokens)

merged_token_table = pd.DataFrame({
    "word_index": np.arange(len(merged_tokens)),
    "merged_token": merged_tokens,
    "subword_indices": [",".join(map(str, group)) for group in subword_groups],
    "original_subwords": [
        " | ".join(subword_tokens[index] for index in group)
        for group in subword_groups
    ],
    "number_of_subwords": [len(group) for group in subword_groups]
})

print("Subword count:", len(subword_tokens))
print("Merged-token count:", len(merged_tokens))
display(merged_token_table.head(50))

## 13. Aggregate subword attention to merged-token attention

In [ ]:
def aggregate_attention_to_words(subword_attention, groups):
    word_count = len(groups)
    word_attention = np.zeros((word_count, word_count), dtype=np.float64)

    for source_word, source_indices in enumerate(groups):
        for target_word, target_indices in enumerate(groups):
            block = subword_attention[np.ix_(source_indices, target_indices)]
            word_attention[source_word, target_word] = float(block.mean())

    return word_attention


merged_attention = aggregate_attention_to_words(
    mean_subword_attention,
    subword_groups
)

assert merged_attention.shape == (len(merged_tokens), len(merged_tokens))
assert np.isfinite(merged_attention).all()
assert (merged_attention >= 0).all()

print("Merged attention shape:", merged_attention.shape)
print("Minimum weight:", merged_attention.min())
print("Maximum weight:", merged_attention.max())

## 14. Construct the sparse directed weighted graph

In [ ]:
def is_punctuation_only(token):
    text = str(token).strip()
    return (not text) or (not any(character.isalnum() for character in text))


def build_attention_graph(tokens, attention_matrix, top_k):
    graph = nx.DiGraph()
    retained_node_ids = []

    for node_id, token in enumerate(tokens):
        if REMOVE_SPECIAL_TOKENS and token in SPECIAL_TOKENS:
            continue

        if REMOVE_PUNCTUATION_ONLY_TOKENS and is_punctuation_only(token):
            continue

        retained_node_ids.append(node_id)
        graph.add_node(
            int(node_id),
            token=str(token),
            position=int(node_id)
        )

    for source_id in retained_node_ids:
        candidates = []

        for target_id in retained_node_ids:
            if REMOVE_SELF_LOOPS and source_id == target_id:
                continue

            weight = float(attention_matrix[source_id, target_id])

            if not np.isfinite(weight) or weight <= 0:
                continue

            candidates.append((target_id, weight))

        candidates.sort(key=lambda item: (-item[1], item[0]))

        for target_id, weight in candidates[:top_k]:
            graph.add_edge(
                int(source_id),
                int(target_id),
                weight=weight,
                attention=weight
            )

    return graph


G = build_attention_graph(
    merged_tokens,
    merged_attention,
    TOP_K_EDGES_PER_NODE
)

if G.number_of_nodes() == 0:
    raise RuntimeError("The graph contains no nodes.")

if G.number_of_edges() == 0:
    raise RuntimeError("The graph contains no edges.")

print("Graph type:", type(G).__name__)
print("Directed:", G.is_directed())
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Density:", nx.density(G))

## 15. Create node, edge, and graph-statistics tables

In [ ]:
node_table = pd.DataFrame([
    {
        "node_id": node_id,
        "token": data["token"],
        "position": data["position"],
        "in_degree": G.in_degree(node_id),
        "out_degree": G.out_degree(node_id),
        "weighted_in_degree": G.in_degree(node_id, weight="weight"),
        "weighted_out_degree": G.out_degree(node_id, weight="weight")
    }
    for node_id, data in G.nodes(data=True)
]).sort_values("node_id").reset_index(drop=True)

edge_table = pd.DataFrame([
    {
        "source_id": source,
        "source_token": G.nodes[source]["token"],
        "target_id": target,
        "target_token": G.nodes[target]["token"],
        "attention_weight": data["weight"]
    }
    for source, target, data in G.edges(data=True)
]).sort_values(
    ["attention_weight", "source_id", "target_id"],
    ascending=[False, True, True]
).reset_index(drop=True)

weak_components = list(nx.weakly_connected_components(G))
strong_components = list(nx.strongly_connected_components(G))

graph_statistics = {
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "split": DATASET_SPLIT,
    "sample_index": SAMPLE_INDEX,
    "subword_tokens": len(subword_tokens),
    "merged_tokens_before_filtering": len(merged_tokens),
    "graph_nodes": G.number_of_nodes(),
    "graph_edges": G.number_of_edges(),
    "graph_density": nx.density(G),
    "average_in_degree": np.mean([degree for _, degree in G.in_degree()]),
    "average_out_degree": np.mean([degree for _, degree in G.out_degree()]),
    "maximum_in_degree": max(degree for _, degree in G.in_degree()),
    "maximum_out_degree": max(degree for _, degree in G.out_degree()),
    "weakly_connected_components": len(weak_components),
    "strongly_connected_components": len(strong_components),
    "largest_weak_component_size": max(map(len, weak_components)),
    "largest_strong_component_size": max(map(len, strong_components)),
    "top_k_edges_per_node": TOP_K_EDGES_PER_NODE,
    "attention_extraction_seconds": elapsed_seconds
}

graph_statistics_table = pd.DataFrame([graph_statistics])

display(node_table.head(20))
display(edge_table.head(20))
display(graph_statistics_table.T.rename(columns={0: "value"}))

## 16. Export Notebook 1 results

In [ ]:
subword_table.to_csv(OUTPUT_DIR / "subword_tokens.csv", index=False)
merged_token_table.to_csv(OUTPUT_DIR / "merged_tokens.csv", index=False)
node_table.to_csv(OUTPUT_DIR / "graph_nodes.csv", index=False)
edge_table.to_csv(OUTPUT_DIR / "graph_edges.csv", index=False)
graph_statistics_table.to_csv(OUTPUT_DIR / "graph_statistics.csv", index=False)

attention_labels = [
    f"{index}:{token}" for index, token in enumerate(merged_tokens)
]

pd.DataFrame(
    merged_attention,
    index=attention_labels,
    columns=attention_labels
).to_csv(OUTPUT_DIR / "merged_attention_matrix.csv")

np.savez_compressed(
    OUTPUT_DIR / "attention_data.npz",
    mean_subword_attention=mean_subword_attention,
    merged_attention=merged_attention,
    subword_tokens=np.array(subword_tokens, dtype=object),
    merged_tokens=np.array(merged_tokens, dtype=object),
    subword_groups=np.array(subword_groups, dtype=object)
)

nx.write_graphml(G, OUTPUT_DIR / "attention_graph.graphml")

with open(OUTPUT_DIR / "attention_graph.gpickle", "wb") as file:
    pickle.dump(G, file)

print("All data files saved in:", OUTPUT_DIR)

## 17. Attention heatmap

In [ ]:
heatmap_limit = min(50, len(merged_tokens))

plt.figure(figsize=(14, 12))
plt.imshow(
    merged_attention[:heatmap_limit, :heatmap_limit],
    aspect="auto"
)
plt.colorbar(label="Mean attention weight")
plt.xticks(
    range(heatmap_limit),
    merged_tokens[:heatmap_limit],
    rotation=90,
    fontsize=7
)
plt.yticks(
    range(heatmap_limit),
    merged_tokens[:heatmap_limit],
    fontsize=7
)
plt.xlabel("Attended-to token")
plt.ylabel("Source/query token")
plt.title("BioGPT Word-Level Attention Matrix")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "attention_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## 18. Sparse attention-graph figure

In [ ]:
number_of_visual_edges = min(80, G.number_of_edges())

strongest_edges = sorted(
    G.edges(data=True),
    key=lambda item: item[2]["weight"],
    reverse=True
)[:number_of_visual_edges]

visual_graph = nx.DiGraph()

for source, target, data in strongest_edges:
    visual_graph.add_node(source, **G.nodes[source])
    visual_graph.add_node(target, **G.nodes[target])
    visual_graph.add_edge(source, target, **data)

layout = nx.spring_layout(visual_graph, seed=SEED, k=1.5)

labels = {
    node_id: visual_graph.nodes[node_id]["token"]
    for node_id in visual_graph.nodes()
}

node_sizes = [
    300 + 5000 * visual_graph.in_degree(node_id, weight="weight")
    for node_id in visual_graph.nodes()
]

edge_widths = [
    0.5 + 15 * visual_graph.edges[source, target]["weight"]
    for source, target in visual_graph.edges()
]

plt.figure(figsize=(18, 14))
nx.draw_networkx_nodes(
    visual_graph,
    layout,
    node_size=node_sizes,
    alpha=0.85
)
nx.draw_networkx_edges(
    visual_graph,
    layout,
    width=edge_widths,
    alpha=0.45,
    arrows=True,
    arrowsize=12,
    connectionstyle="arc3,rad=0.08"
)
nx.draw_networkx_labels(
    visual_graph,
    layout,
    labels=labels,
    font_size=8
)
plt.title("BioGPT Clinical Attention Graph")
plt.axis("off")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "attention_graph.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## 19. Create one ZIP archive

In [ ]:
archive_path = shutil.make_archive(
    "/content/notebook1_outputs",
    "zip",
    root_dir=OUTPUT_DIR
)

print("ZIP archive:", archive_path)

# To download from Colab, uncomment:
# from google.colab import files
# files.download(archive_path)

## 20. Final validation

In [ ]:
required_files = [
    "metadata.json",
    "subword_tokens.csv",
    "merged_tokens.csv",
    "graph_nodes.csv",
    "graph_edges.csv",
    "graph_statistics.csv",
    "merged_attention_matrix.csv",
    "attention_data.npz",
    "attention_graph.graphml",
    "attention_graph.gpickle",
    "attention_heatmap.png",
    "attention_graph.png"
]

missing_files = [
    filename
    for filename in required_files
    if not (OUTPUT_DIR / filename).exists()
]

if missing_files:
    raise RuntimeError(f"Missing outputs: {missing_files}")

print("=" * 70)
print("NOTEBOOK 1 COMPLETED SUCCESSFULLY")
print("=" * 70)
print("Question ID:", sample["id"])
print("Correct answer:", correct_letter, "-", correct_answer)
print("Merged tokens:", len(merged_tokens))
print("Graph nodes:", G.number_of_nodes())
print("Graph edges:", G.number_of_edges())
print("Graph density:", nx.density(G))
print("Output folder:", OUTPUT_DIR)
print("ZIP archive:", archive_path)
print("=" * 70)
print("Use attention_graph.gpickle and metadata.json in Notebook 2.")